
# Create DOST-PCAARRD Awards from Official Transparency Seal GIA PDFs

Creates OpenAlex award rows for the Philippine Council for Agriculture, Aquatic and Natural Resources Research and Development (DOST-PCAARRD) Grants-in-Aid project disclosures.

**Awarding body:** Philippine Council for Agriculture, Aquatic and Natural Resources Research and Development - `F4320336119` (PH; no DOI/ROR in the OpenAlex public API result).

**Source:** official DOST-PCAARRD Transparency Seal page and its yearly `DOST-PCAARRD List of Grants in Aid (GIA) Projects` PDFs for 2016-2025.

**Schema choices:**
- One row per deduplicated official project. Yearly disclosure repeats are collapsed by source-stable project title, implementing agency, and start/end date.
- `amount` is total project cost in PHP. Annual PCAARRD GIA values remain in `gia_by_source_year_json` for audit.
- `funding_type = 'grant'`; `funder_scheme = program_title` when the PDF layout exposes it.
- PI names are not published in these Transparency Seal tables. `lead_investigator` is organization-shaped only when a clean implementing agency can be parsed.

**S3 location:** `s3a://openalex-ingest/awards/pcaarrd/pcaarrd_gia_projects.parquet`

**Local full-source validation on 2026-05-28:** 3,144 deduplicated project rows from 3,424 annual disclosure rows; 0 duplicate `funder_award_id`s; start years 2012-2025; 99.6% amount coverage; total PHP 22,351,403,518.06.

Contractor has no S3/Databricks access; admin must upload the parquet, run this notebook, inspect verification cells, and decide later whether a job YAML is warranted.


## Step 1: Create Raw Table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.pcaarrd_awards_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/pcaarrd/pcaarrd_gia_projects.parquet`;


In [ ]:
%sql
-- Local full-source validation on 2026-05-28 found 3,144 deduplicated projects.
SELECT COUNT(*) AS raw_rows
FROM openalex.awards.pcaarrd_awards_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.pcaarrd_awards_raw;


In [ ]:
%sql
SELECT
    funder_award_id,
    display_name,
    program_title,
    source_implementing_agency,
    start_date,
    end_date,
    amount,
    currency,
    status,
    source_years,
    source_pdf_url
FROM openalex.awards.pcaarrd_awards_raw
LIMIT 10;


## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys


In [ ]:
%sql
-- Money-field scan per runbook Step 1.5.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'pcaarrd_awards_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant|gia|award';


In [ ]:
%sql
-- Native key uniqueness: must return zero rows.
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.pcaarrd_awards_raw
GROUP BY funder_award_id
HAVING COUNT(*) > 1
ORDER BY rows DESC, funder_award_id
LIMIT 20;


In [ ]:
%sql
-- Core coverage check.
SELECT
    COUNT(*) AS rows,
    COUNT(display_name) AS has_title,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    COUNT(source_implementing_agency) AS has_implementing_agency,
    ROUND(try_divide(COUNT(source_implementing_agency), COUNT(*)) * 100.0, 1) AS pct_implementing_agency,
    COUNT(start_date) AS has_start_date,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_start_date,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(currency) AS has_currency,
    ROUND(try_divide(COUNT(currency), COUNT(*)) * 100.0, 1) AS pct_currency
FROM openalex.awards.pcaarrd_awards_raw;


In [ ]:
%sql
-- Amount distribution. Amount is total project cost in PHP; annual GIA remains in gia_by_source_year_json.
SELECT
    currency,
    COUNT(*) AS rows,
    COUNT(amount) AS rows_with_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 2) AS total_amount,
    ROUND(MIN(TRY_CAST(amount AS DOUBLE)), 2) AS min_amount,
    ROUND(percentile_approx(TRY_CAST(amount AS DOUBLE), 0.5), 2) AS median_amount,
    ROUND(MAX(TRY_CAST(amount AS DOUBLE)), 2) AS max_amount
FROM openalex.awards.pcaarrd_awards_raw
GROUP BY currency
ORDER BY rows DESC;


In [ ]:
%sql
-- Date, status, and source-year shape.
SELECT
    MIN(TRY_TO_DATE(start_date, 'yyyy-MM-dd')) AS min_start_date,
    MAX(TRY_TO_DATE(start_date, 'yyyy-MM-dd')) AS max_start_date,
    MIN(TRY_TO_DATE(end_date, 'yyyy-MM-dd')) AS min_end_date,
    MAX(TRY_TO_DATE(end_date, 'yyyy-MM-dd')) AS max_end_date,
    COUNT(DISTINCT status) AS distinct_statuses,
    COUNT(DISTINCT source_year) AS distinct_latest_source_years
FROM openalex.awards.pcaarrd_awards_raw;


In [ ]:
%sql
SELECT status, COUNT(*) AS rows
FROM openalex.awards.pcaarrd_awards_raw
GROUP BY status
ORDER BY rows DESC, status;


In [ ]:
%sql
SELECT source_year, COUNT(*) AS rows
FROM openalex.awards.pcaarrd_awards_raw
GROUP BY source_year
ORDER BY source_year;


In [ ]:
%sql
SELECT program_title, COUNT(*) AS rows
FROM openalex.awards.pcaarrd_awards_raw
WHERE program_title IS NOT NULL
GROUP BY program_title
ORDER BY rows DESC, program_title
LIMIT 25;


## Step 1.6: Funder Existence Assertion


In [ ]:
%sql
-- Public OpenAlex API pre-flight on 2026-05-27 verified F4320336119:
-- Philippine Council for Agriculture, Aquatic and Natural Resources Research and Development; country PH; no DOI/ROR.
WITH expected AS (
    SELECT
        4320336119 AS expected_funder_id,
        'Philippine Council for Agriculture, Aquatic and Natural Resources Research and Development' AS expected_display_name
),
raw_funders AS (
    SELECT
        COUNT(DISTINCT TRY_CAST(funder_id AS BIGINT)) AS distinct_raw_funder_ids,
        MIN(TRY_CAST(funder_id AS BIGINT)) AS min_raw_funder_id,
        MAX(TRY_CAST(funder_id AS BIGINT)) AS max_raw_funder_id
    FROM openalex.awards.pcaarrd_awards_raw
)
SELECT
    CASE
        WHEN r.distinct_raw_funder_ids = 1
         AND r.min_raw_funder_id = e.expected_funder_id
         AND r.max_raw_funder_id = e.expected_funder_id
        THEN 'OK'
        ELSE raise_error(CONCAT(
            'PCAARRD raw funder_id mismatch: distinct=', r.distinct_raw_funder_ids,
            ', min=', r.min_raw_funder_id,
            ', max=', r.max_raw_funder_id,
            ', expected=', e.expected_funder_id
        ))
    END AS funder_assertion,
    e.expected_funder_id AS funder_id,
    e.expected_display_name AS display_name,
    'https://openalex.org/F4320336119' AS openalex_funder_url
FROM expected e
CROSS JOIN raw_funders r;


## Step 2: Transform to OpenAlex Awards Schema


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.pcaarrd_awards
USING delta
AS
WITH normalized AS (
    SELECT
        *,
        TRY_CAST(funder_id AS BIGINT) AS funder_id_bigint,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS start_date_parsed,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS end_date_parsed,
        TRY_CAST(amount AS DOUBLE) AS amount_double,
        NULLIF(TRIM(source_implementing_agency), '') AS implementing_agency_clean
    FROM openalex.awards.pcaarrd_awards_raw
    WHERE funder_award_id IS NOT NULL
      AND display_name IS NOT NULL
)
SELECT
    abs(xxhash64(CONCAT(TRY_CAST(funder_id_bigint AS STRING), ':', LOWER(funder_award_id)))) % 9000000000 AS id,
    display_name,
    description,
    funder_id_bigint AS funder_id,
    funder_award_id,
    amount_double AS amount,
    currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(funder_id_bigint AS STRING)) AS id,
        'Philippine Council for Agriculture, Aquatic and Natural Resources Research and Development' AS display_name,
        CAST(NULL AS STRING) AS ror_id,
        CAST(NULL AS STRING) AS doi
    ) AS funder,
    'grant' AS funding_type,
    program_title AS funder_scheme,
    'pcaarrd_gia_projects' AS provenance,
    start_date_parsed AS start_date,
    end_date_parsed AS end_date,
    YEAR(start_date_parsed) AS start_year,
    YEAR(end_date_parsed) AS end_year,
    CASE
        WHEN implementing_agency_clean IS NULL THEN NULL
        ELSE struct(
            CAST(NULL AS STRING) AS given_name,
            CAST(NULL AS STRING) AS family_name,
            CAST(NULL AS STRING) AS orcid,
            start_date_parsed AS role_start,
            struct(
                implementing_agency_clean AS name,
                'PH' AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    source_pdf_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(funder_id_bigint AS STRING), ':', LOWER(funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    CAST(NULL AS BOOLEAN) AS declined,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM normalized;


In [ ]:
%sql
SELECT COUNT(*) AS transformed_rows
FROM openalex.awards.pcaarrd_awards;


In [ ]:
%sql
-- Confirm generated OpenAlex award IDs are unique.
SELECT id, COUNT(*) AS rows
FROM openalex.awards.pcaarrd_awards
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY rows DESC, id
LIMIT 20;


## Step 3: Delete Existing Priority Slice and Insert


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'pcaarrd_gia_projects' AND priority = 160;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    160 AS priority
FROM openalex.awards.pcaarrd_awards;


## Step 6: Verification


In [ ]:
%sql
SELECT
    'raw' AS table_name,
    COUNT(*) AS rows
FROM openalex.awards.pcaarrd_awards_raw
UNION ALL
SELECT
    'transformed' AS table_name,
    COUNT(*) AS rows
FROM openalex.awards.pcaarrd_awards
UNION ALL
SELECT
    'openalex_awards_raw_priority_160' AS table_name,
    COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'pcaarrd_gia_projects'
  AND priority = 160;


In [ ]:
%sql
SELECT
    COUNT(*) AS rows,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(SUM(amount), 2) AS total_amount,
    MIN(start_year) AS min_start_year,
    MAX(start_year) AS max_start_year,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'pcaarrd_gia_projects'
  AND priority = 160;


In [ ]:
%sql
SELECT
    id,
    display_name,
    amount,
    currency,
    funder_scheme,
    start_date,
    end_date,
    lead_investigator.affiliation.name AS implementing_agency,
    landing_page_url
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'pcaarrd_gia_projects'
  AND priority = 160
ORDER BY start_date DESC, display_name
LIMIT 25;


In [ ]:
%sql
-- Dedup safety inside this provenance/priority slice: must return zero rows.
SELECT id, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'pcaarrd_gia_projects'
  AND priority = 160
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY rows DESC, id
LIMIT 20;



## Handoff Notes

Contractor-complete state stops here. Admin must:

- run `scripts/local/pcaarrd_to_s3.py --allow-shrink` only after reviewing any future shrink, otherwise run without override;
- upload the parquet to `s3a://openalex-ingest/awards/pcaarrd/pcaarrd_gia_projects.parquet`;
- run this notebook on Databricks;
- inspect the Step 6 verification cells;
- only then decide whether to add scheduled job YAML.
